

AutoGen Core - Distributed




In [1]:
from dataclasses import dataclass
from autogen_core import AgentId, MessageContext, RoutedAgent, message_handler
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.messages import TextMessage
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_ext.tools.langchain import LangChainToolAdapter
from langchain_community.utilities import GoogleSerperAPIWrapper
from langchain.agents import Tool
from IPython.display import display, Markdown

from dotenv import load_dotenv

load_dotenv(override=True)

ALL_IN_ONE_WORKER = False

In [2]:

@dataclass
class Message:
    content: str

### And now - a host for distributed runtime

In [3]:
from autogen_ext.runtimes.grpc import GrpcWorkerAgentRuntimeHost

host = GrpcWorkerAgentRuntimeHost(address="localhost:50051")
host.start() 

### Let's reintroduce a tool

In [4]:
serper = GoogleSerperAPIWrapper()
langchain_serper =Tool(name="internet_search", func=serper.run, description="Useful for when you need to search the internet")
autogen_serper = LangChainToolAdapter(langchain_serper)

In [5]:
instruction1 = "To help with a decision on whether to use AutoGen in a new AI Agent project, \
please research and briefly respond with reasons in favor of choosing AutoGen; the pros of AutoGen."

instruction2 = "To help with a decision on whether to use AutoGen in a new AI Agent project, \
please research and briefly respond with reasons against choosing AutoGen; the cons of Autogen."

judge = "You must make a decision on whether to use AutoGen for a project. \
Your research team has come up with the following reasons for and against. \
Based purely on the research from your team, please respond with your decision and brief rationale."

### And make some Agents

In [6]:
class Player1Agent(RoutedAgent):
    def __init__(self, name: str) -> None:
        super().__init__(name)
        model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")
        self._delegate = AssistantAgent(name, model_client=model_client, tools=[autogen_serper], reflect_on_tool_use=True)

    @message_handler
    async def handle_my_message_type(self, message: Message, ctx: MessageContext) -> Message:
        text_message = TextMessage(content=message.content, source="user")
        response = await self._delegate.on_messages([text_message], ctx.cancellation_token)
        return Message(content=response.chat_message.content)
    
class Player2Agent(RoutedAgent):
    def __init__(self, name: str) -> None:
        super().__init__(name)
        model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")
        self._delegate = AssistantAgent(name, model_client=model_client, tools=[autogen_serper], reflect_on_tool_use=True)

    @message_handler
    async def handle_my_message_type(self, message: Message, ctx: MessageContext) -> Message:
        text_message = TextMessage(content=message.content, source="user")
        response = await self._delegate.on_messages([text_message], ctx.cancellation_token)
        return Message(content=response.chat_message.content)
    
class Judge(RoutedAgent):
    def __init__(self, name: str) -> None:
        super().__init__(name)
        model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")
        self._delegate = AssistantAgent(name, model_client=model_client)
        
    @message_handler
    async def handle_my_message_type(self, message: Message, ctx: MessageContext) -> Message:
        message1 = Message(content=instruction1)
        message2 = Message(content=instruction2)
        inner_1 = AgentId("player1", "default")
        inner_2 = AgentId("player2", "default")
        response1 = await self.send_message(message1, inner_1)
        response2 = await self.send_message(message2, inner_2)
        result = f"## Pros of AutoGen:\n{response1.content}\n\n## Cons of AutoGen:\n{response2.content}\n\n"
        judgement = f"{judge}\n{result}Respond with your decision and brief explanation"
        message = TextMessage(content=judgement, source="user")
        response = await self._delegate.on_messages([message], ctx.cancellation_token)
        return Message(content=result + "\n\n## Decision:\n\n" + response.chat_message.content)


In [7]:
from autogen_ext.runtimes.grpc import GrpcWorkerAgentRuntime

if ALL_IN_ONE_WORKER:

    worker = GrpcWorkerAgentRuntime(host_address="localhost:50051")
    await worker.start()

    await Player1Agent.register(worker, "player1", lambda: Player1Agent("player1"))
    await Player2Agent.register(worker, "player2", lambda: Player2Agent("player2"))
    await Judge.register(worker, "judge", lambda: Judge("judge"))

    agent_id = AgentId("judge", "default")

else:

    worker1 = GrpcWorkerAgentRuntime(host_address="localhost:50051")
    await worker1.start()
    await Player1Agent.register(worker1, "player1", lambda: Player1Agent("player1"))

    worker2 = GrpcWorkerAgentRuntime(host_address="localhost:50051")
    await worker2.start()
    await Player2Agent.register(worker2, "player2", lambda: Player2Agent("player2"))

    worker = GrpcWorkerAgentRuntime(host_address="localhost:50051")
    await worker.start()
    await Judge.register(worker, "judge", lambda: Judge("judge"))
    agent_id = AgentId("judge", "default")




In [8]:
response = await worker.send_message(Message(content="Go!"), agent_id)

In [9]:
display(Markdown(response.content))

## Pros of AutoGen:
Here are some key reasons in favor of choosing AutoGen for your AI Agent project:

1. **Open-Source Framework**: AutoGen is an open-source framework, which allows for flexibility in customization and community support for ongoing improvements and features.

2. **Multi-Agent Capabilities**: It is designed for building multi-agent systems that can operate collaboratively, enhancing problem-solving capabilities by leveraging the strengths of different agents working together.

3. **Efficiency and Scalability**: AutoGen improves the efficiency of AI applications by enabling parallel processing and task distribution among agents, which is particularly beneficial for complex tasks requiring significant computational resources.

4. **Ease of Integration**: The framework simplifies the integration of various AI models and components, making it easier to develop applications that can utilize different methodologies or technologies within a unified system.

5. **Rich Ecosystem**: Being part of the growing ecosystem of AI development tools, AutoGen can benefit from advancements in related technologies and contribute to more robust and innovative AI applications.

6. **Support for Human-AI Collaboration**: The framework is designed to facilitate interactions between agents and human users, promoting collaborative environments where AI can enhance human decision-making processes.

Choosing AutoGen can lead to more capable and adaptable AI agents suited for a variety of applications.

TERMINATE

## Cons of AutoGen:
Here are some cons of using AutoGen for your AI Agent project:

1. **Steep Learning Curve**: Many users find it challenging to navigate AutoGen's complex interface and features, requiring significant time and effort to become proficient.

2. **User Difficulty**: The platform may not be user-friendly, posing difficulties for those who are not experienced with its functionalities or general AI development.

3. **Lack of Intuition**: The interface and workflow might not be intuitive, leading to frustration and inefficiency as users struggle to understand how to best utilize the tools available.

4. **Limited Support and Documentation**: There could be insufficient resources such as tutorials or documentation, making it harder for new users to find solutions to their problems.

5. **Integration Issues**: Depending on the existing tech stack, integration with other systems or technologies might be problematic, leading to delays or added complexity in the project.

These factors could impact the overall efficiency and success of your AI Agent project. 

TERMINATE



## Decision:

After evaluating the pros and cons of using AutoGen for the AI Agent project, I recommend using AutoGen. The advantages, particularly its open-source nature, multi-agent capabilities, and efficiency in handling complex tasks, align well with our project's goals. Notably, the rich ecosystem and support for human-AI collaboration present significant long-term benefits for adaptability and innovation. While the steep learning curve and potential user difficulties are valid concerns, the strategic advantages provided by AutoGen's capabilities outweigh these challenges. Proper training and onboarding can mitigate the learning curve, making it a worthwhile investment. 

TERMINATE

In [10]:
await worker.stop()
if not ALL_IN_ONE_WORKER:
    await worker1.stop()
    await worker2.stop()

In [11]:
await host.stop()